# WNMFDDA — Rejection Hypotheses RH6–RH8

Companion notebook to RH1–RH5, adapted for Kaggle T4 GPU. Iterative NMF updates use PyTorch CUDA when available; completed seed/fold units are checkpointed and skipped on reruns. CPU remains the fallback/reference. It follows the same **E / P / Q / V / L** structure: **E**quations and assumptions, **P**roposition, **Q**uantities and decision rule, **V**erification, and **L**ink to consequences.

- **RH6:** interaction between graph regularization (`lambda`) and factor shrinkage (`beta`).
- **RH7:** true entity-aligned similarity graphs versus node-permuted graphs.
- **RH8:** original unnormalized graph Laplacian versus a normalized-Laplacian objective.

Run top-to-bottom. Start with `PROFILE = "smoke"`; use `quick` for development and `full` for final results. Attach `DataF.mat` and `DataC.mat` or set `DATA_DIR`. Unobserved pairs are treated as negatives, matching the existing benchmark protocol. Iteration caps are not convergence guarantees.

In [1]:
import os, sys, time, glob, warnings
from types import SimpleNamespace as NS
import numpy as np, pandas as pd, scipy.io as sio
from scipy import stats
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
try:
    import torch
except ImportError:
    torch = None
warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    from IPython.display import display
except Exception:
    display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 160)

# ------------------------------ CONFIG ------------------------------
PROFILE = os.environ.get("WNMFDDA_PROFILE", "quick")          # 'smoke' | 'quick' | 'full'
REQUESTED_BACKEND = os.environ.get("WNMFDDA_BACKEND", "auto").lower()  # auto | cuda | cpu
CUDA_AVAILABLE = bool(torch is not None and torch.cuda.is_available())
if REQUESTED_BACKEND == "cuda" and not CUDA_AVAILABLE:
    raise RuntimeError("CUDA requested but unavailable. Enable GPU in Kaggle Notebook Settings.")
DEVICE = torch.device("cuda" if CUDA_AVAILABLE and REQUESTED_BACKEND != "cpu" else "cpu") if torch is not None else None
BACKEND = "torch-cuda" if DEVICE is not None and DEVICE.type == "cuda" else "numpy-cpu"
SESSION_BUDGET_HOURS = float(os.environ.get("WNMFDDA_SESSION_BUDGET_HOURS", "10"))
SESSION_START = time.monotonic()
SESSION_DEADLINE = SESSION_START + SESSION_BUDGET_HOURS * 3600
PROFILES = {
    "smoke": dict(N_SEEDS=2,  N_FOLDS=3,  MAX_ITER=30,  WARM_PARTS=1, DENS=[1.0, 0.25, 0.05]),
    "quick": dict(N_SEEDS=5,  N_FOLDS=5,  MAX_ITER=200, WARM_PARTS=2, DENS=[1.0, 0.5, 0.25, 0.10, 0.05]),
    "full":  dict(N_SEEDS=20, N_FOLDS=10, MAX_ITER=500, WARM_PARTS=3, DENS=[1.0, 0.5, 0.25, 0.10, 0.05]),
}
globals().update(PROFILES[PROFILE])
SEEDS = list(range(N_SEEDS))
WARM_SPLIT = 5                                                # RH1: held-out entities' positives are split into 5 parts for the warm reference

PAPER = dict(p=5, K=5, a=0.5, k=160, lam=1.0, beta=0.02)
ELIG = True                                                   # neighbours restricted to entities with >= 1 training association
DATA_DIR = os.environ.get("DATA_DIR", "/mnt/data")
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RES_PATH = os.path.join(OUT_DIR, f"wnmfdda_rh678_results_{PROFILE}_{BACKEND.replace('-', '_')}.csv")
FORCE_RERUN = os.environ.get("WNMFDDA_FORCE_RERUN", "0") == "1"

# decision-rule constants (fixed before running)
DELTA = 0.01; ALPHA = 0.05
RH1_DROP_MIN, RH1_DROP_FALSIFY, RH1_MARGIN = 0.10, 0.05, 0.05
RH2_DROP_MIN = 0.10
RH3_LIFT_MIN, RH3_GAP_MAX = 0.70, 0.06

# dev-tuning grids for the training-free baselines (tuned on an inner split of the TRAINING matrix only)
KNN_K = [1, 3, 5, 10, 20, 50, 100]; KNN_G = [1.0, 4.0]
DIFF_L = [0.3, 1.0, 3.0, 10.0, 30.0]
SVD_K = [10, 20, 40, 80, 160]

if FORCE_RERUN and os.path.exists(RES_PATH):
    os.remove(RES_PATH)
RES = pd.read_csv(RES_PATH) if (os.path.exists(RES_PATH) and not FORCE_RERUN) else pd.DataFrame()
PRIMARY, VERDICT = {}, {}

# user-supplied pilot numbers (dataset not stated; Fdataset-like). Printed next to this run for comparison only.
PILOT = dict(
    RH1=dict(LDO=dict(W1G1=0.788, warm=0.919, POP=0.734, KNN=0.765), LDiO=dict(W1G1=0.738, KNN=0.699)),
    RH2={1.0: (0.919, 0.054), 0.5: (0.851, 0.059), 0.25: (0.758, 0.046), 0.10: (0.648, 0.028), 0.05: (0.559, 0.004)},
    RH3=dict(sup_pos=0.64, sup_neg=0.06, W1G1_sup=0.986, W1G1_unsup=0.801, WKNN_ONLY_unsup=0.47, ONEBIT=0.784,
             KNN=0.865, W1G1=0.919, POP=0.708, WKNN_ONLY=0.798, POP_unsup=0.65),
)
print(f"PROFILE={PROFILE} seeds={N_SEEDS} folds={N_FOLDS} max_iter={MAX_ITER} warm_parts={WARM_PARTS} densities={DENS}")
print(f"backend={BACKEND}; device={DEVICE}; CUDA available={CUDA_AVAILABLE}")
if CUDA_AVAILABLE: print("GPU:", torch.cuda.get_device_name(0))
print(f"session budget={SESSION_BUDGET_HOURS:.1f} hours; data dir: {DATA_DIR}")
print(f"cached rows: {len(RES)} -> {RES_PATH}")

PROFILE=quick seeds=5 folds=5 max_iter=200 warm_parts=2 densities=[1.0, 0.5, 0.25, 0.1, 0.05]
backend=torch-cuda; device=cuda; CUDA available=True
GPU: Tesla T4
session budget=10.0 hours; data dir: /mnt/data
cached rows: 0 -> /kaggle/working/wnmfdda_rh678_results_quick_torch_cuda.csv


In [4]:
# ============================== CORE LIBRARY ==============================
EPS = 1e-12

def neighbor_order(S):
    S2 = np.array(S, dtype=float); np.fill_diagonal(S2, -np.inf)
    return np.argsort(-S2, axis=1, kind="stable")[:, :-1]          # neighbours by descending similarity, self removed

def build_graph(S, p):
    # published rule: W=1 if mutual p-NN, 0 if neither selects the other, 0.5 otherwise;  S* = S (.) W
    n = S.shape[0]; order = neighbor_order(S)
    N = np.zeros((n, n), bool); N[np.arange(n)[:, None], order[:, :p]] = True
    W = np.where(N & N.T, 1.0, np.where(~(N | N.T), 0.0, 0.5)); np.fill_diagonal(W, 0.0)
    Sg = S * W; np.fill_diagonal(Sg, 0.0)
    return Sg

def wknn_side(S, P, K, a, order, eligible=None, gamma=1.0):
    # rows of P are the profiles being propagated.  numerator: a^(t-1) * s^gamma ; denominator: sum of the selected s^gamma
    # (paper: gamma=1, a=0.5 -> decay only in the numerator).  a=1 gives a plain similarity-weighted kNN average.
    eligible = ELIG if eligible is None else eligible
    n = S.shape[0]; rows = np.arange(n)[:, None]
    valid = (P.sum(axis=1) > 0)[order] if eligible else np.ones(order.shape, bool)
    cum = np.cumsum(valid, axis=1)
    sel = valid & (cum <= K)
    sims = np.maximum(S[rows, order], 0.0) ** gamma
    w = np.where(sel, np.power(a, np.maximum(cum - 1, 0)) * sims, 0.0)
    den = np.where(sel, sims, 0.0).sum(axis=1)
    Wf = np.zeros((n, n)); Wf[rows, order] = w
    out = Wf @ P
    ok = den > EPS
    out[ok] /= den[ok, None]; out[~ok] = 0.0
    return out

def wknn_reconstruct(Y, D, K=None, a=None):
    K = PAPER["K"] if K is None else K; a = PAPER["a"] if a is None else a
    Yr = wknn_side(D.SR, Y, K, a, D.ordR)
    Yd = wknn_side(D.SD, Y.T, K, a, D.ordD).T
    return np.maximum(Y, (Yr + Yd) / 2), Yr, Yd                     # Y' = max(Y, (Y_r + Y_d)/2)

def knn_score(Y, D, K, gamma):
    Yr = wknn_side(D.SR, Y, K, 1.0, D.ordR, gamma=gamma)
    Yd = wknn_side(D.SD, Y.T, K, 1.0, D.ordD, gamma=gamma).T
    return (Yr + Yd) / 2

def diffusion_score(Y, D, lam):
    # closed-form graph diffusion (I + lam L)^-1 applied to each side, averaged; no factorization, no iterative training
    lr, Vr = D.eigR; ld, Vd = D.eigD
    Fr = Vr @ ((Vr.T @ Y) / (1.0 + lam * lr)[:, None])
    Fd = ((Y @ Vd) / (1.0 + lam * ld)[None, :]) @ Vd.T
    return (Fr + Fd) / 2

def svd_score(Yp, k):
    U, s, Vt = np.linalg.svd(Yp, full_matrices=False)
    return (U[:, :k] * s[:k]) @ Vt[:k]

def pop_product(Y):  return np.outer(Y.sum(1), Y.sum(0))           # warm popularity: deg(drug) * deg(disease)
def pop_axis(Y, axis):                                              # cold popularity: new drug -> disease popularity, new disease -> drug popularity
    n, m = Y.shape
    return np.tile(Y.sum(0), (n, 1)) if axis == 0 else np.tile(Y.sum(1)[:, None], (1, m))

def nmf_fit(M, SgR, SgD, k, lam, beta, max_iter, seed, tol=1e-4):
    # multiplicative updates, paper Eq. (21)-(22); A: k x n, B: k x m; official column-change stopping rule + iteration cap
    rng = np.random.default_rng(seed); n, m = M.shape
    c = 2 * np.sqrt(max(M.mean(), 1e-8) / k)
    A = rng.random((k, n)) * c + 1e-6; B = rng.random((k, m)) * c + 1e-6
    dR, dD = SgR.sum(1)[None, :], SgD.sum(1)[None, :]
    for it in range(1, max_iter + 1):
        An = A * (B @ M.T + lam * (A @ SgR)) / (beta * A + lam * A * dR + (B @ B.T) @ A + EPS)
        Bn = B * (An @ M + lam * (B @ SgD)) / (beta * B + lam * B * dD + (An @ An.T) @ B + EPS)
        err = max(((An - A) ** 2).sum(0).max(), ((Bn - B) ** 2).sum(0).max())
        A, B = An, Bn
        if err < tol:
            break
    return A, B, it

def wnmfdda_scores(Ytr, D, wknn=True, lam=None, k=None, seed=0, max_iter=None):
    M = wknn_reconstruct(Ytr, D)[0] if wknn else Ytr
    A, B, it = nmf_fit(M, D.SgR, D.SgD, PAPER["k"] if k is None else k, PAPER["lam"] if lam is None else lam,
                       PAPER["beta"], MAX_ITER if max_iter is None else max_iter, seed)
    return A.T @ B, it

# ------------------------------ metrics ------------------------------
def auc_vs(ps, nss):                                                 # AUC of positive scores ps against SORTED negative scores nss (ties = 0.5)
    lo = np.searchsorted(nss, ps, "left"); hi = np.searchsorted(nss, ps, "right")
    return float(np.mean((lo + 0.5 * (hi - lo)) / len(nss))) if len(ps) else np.nan

def metrics(Sc, pos, neg, with_aupr=True):
    s = Sc.ravel(); ps, ns = s[pos], s[neg]
    out = {"auc": auc_vs(ps, np.sort(ns))}
    if with_aupr:
        out["aupr"] = float(average_precision_score(np.r_[np.ones(len(ps)), np.zeros(len(ns))], np.r_[ps, ns]))
    return out

# ------------------------------ splits ------------------------------
def pair_split(D, seed, fold):
    pos = np.argwhere(D.Y > 0)
    test = pos[np.array_split(np.random.default_rng(seed).permutation(len(pos)), N_FOLDS)[fold]]
    Ytr = D.Y.copy(); Ytr[test[:, 0], test[:, 1]] = 0.0              # held-out positives removed BEFORE any label-based step
    return NS(Ytr=Ytr, pos=test[:, 0] * D.m + test[:, 1], neg=D.neg_flat)

def thin_train(Ytr, f, seed, fold):                                   # keep a nested random fraction f of the training positives
    if f >= 1.0:
        return Ytr
    pos = np.argwhere(Ytr > 0)
    order = np.random.default_rng([seed, fold, 777]).permutation(len(pos))
    drop = pos[order[int(round(f * len(pos))):]]
    Y2 = Ytr.copy(); Y2[drop[:, 0], drop[:, 1]] = 0.0
    return Y2

def entity_split(D, axis, seed, fold):
    # axis 0: new drugs (rows removed); axis 1: new diseases (columns removed)
    N = D.n if axis == 0 else D.m
    held = np.sort(np.array_split(np.random.default_rng([seed, 4242, axis]).permutation(N), N_FOLDS)[fold])
    mask = np.zeros(D.Y.shape, bool)
    if axis == 0: mask[held, :] = True
    else:         mask[:, held] = True
    Ycold = np.where(mask, 0.0, D.Y)
    pos = np.flatnonzero((mask & (D.Y > 0)).ravel()); neg = np.flatnonzero((mask & (D.Y == 0)).ravel())
    rp = np.random.default_rng([seed, fold, axis, 555]).permutation(len(pos))
    parts = [pos[p] for p in np.array_split(rp, WARM_SPLIT)]
    return NS(Ycold=Ycold, held=held, mask=mask, pos=pos, neg=neg, parts=parts)

# ------------------------------ dev tuning (training matrix only) ------------------------------
def dev_split(Ytr, rng, mode, frac=0.2):
    n, m = Ytr.shape
    if mode == "pair":
        pos = np.argwhere(Ytr > 0)
        held = pos[rng.permutation(len(pos))[:max(1, int(round(frac * len(pos))))]]
        Yin = Ytr.copy(); Yin[held[:, 0], held[:, 1]] = 0.0
        return Yin, held[:, 0] * m + held[:, 1], np.flatnonzero(Ytr.ravel() == 0)
    ax = 0 if mode == "drug" else 1
    ent = np.flatnonzero((Ytr.sum(1) if ax == 0 else Ytr.sum(0)) > 0)
    h = rng.choice(ent, size=max(1, int(round(frac * len(ent)))), replace=False)
    mask = np.zeros(Ytr.shape, bool)
    if ax == 0: mask[h, :] = True
    else:       mask[:, h] = True
    return (np.where(mask, 0.0, Ytr), np.flatnonzero((mask & (Ytr > 0)).ravel()), np.flatnonzero((mask & (Ytr == 0)).ravel()))

def tune(make, grid, Ytr, key, mode):
    Yin, pos, neg = dev_split(Ytr, np.random.default_rng(list(key)), mode)
    f = make(Yin); best, bs = grid[0], -1.0
    for p in grid:
        a = metrics(f(p), pos, neg, False)["auc"]
        if a > bs: best, bs = p, a
    return best, bs

def tune_knn(Ytr, D, key, mode):
    return tune(lambda Yin: (lambda p: knn_score(Yin, D, p[0], p[1])), [(K, g) for K in KNN_K for g in KNN_G], Ytr, key, mode)

def tune_diff(Ytr, D, key, mode):
    return tune(lambda Yin: (lambda lam: diffusion_score(Yin, D, lam)), DIFF_L, Ytr, key, mode)

def tune_svd(Ytr, D, key, mode):
    def make(Yin):
        U, s, Vt = np.linalg.svd(wknn_reconstruct(Yin, D)[0], full_matrices=False)
        return lambda k: (U[:, :k] * s[:k]) @ Vt[:k]
    return tune(make, [k for k in SVD_K if k <= min(D.n, D.m)], Ytr, key, mode)

# ------------------------------ statistics ------------------------------
def pstats(d, alpha=ALPHA):
    d = np.asarray(d, float); d = d[np.isfinite(d)]; n = len(d)
    o = dict(n=n, mean=np.nan, sd=np.nan, ci_lo=np.nan, ci_hi=np.nan, min=np.nan, max=np.nan)
    if n == 0: return o
    o.update(mean=d.mean(), min=d.min(), max=d.max())
    if n >= 2:
        sd = d.std(ddof=1); se = sd / np.sqrt(n); tc = stats.t.ppf(1 - alpha / 2, n - 1)
        o.update(sd=sd, ci_lo=o["mean"] - tc * se, ci_hi=o["mean"] + tc * se)
    return o

def crow(label, d):
    s = pstats(d)
    return dict(contrast=label, n_seeds=s["n"], mean=s["mean"], ci95_lo=s["ci_lo"], ci95_hi=s["ci_hi"], worst=s["min"], best=s["max"])

def one_sided_p(d, thr, alt):
    # alt='gt': H1 mean > thr ; alt='lt': H1 mean < thr  (paired t over seed-level values)
    d = np.asarray(d, float); d = d[np.isfinite(d)]; n = len(d)
    if n < 2: return np.nan
    se = d.std(ddof=1) / np.sqrt(n)
    if se == 0:
        return 0.0 if ((d.mean() > thr) if alt == "gt" else (d.mean() < thr)) else 1.0
    t = (d.mean() - thr) / se
    return float(stats.t.sf(t, n - 1) if alt == "gt" else stats.t.cdf(t, n - 1))

def holm(p):
    p = np.nan_to_num(np.asarray(p, float), nan=1.0); m = len(p)
    order = np.argsort(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def vstr(sup, fal):
    return "SUPPORTED" if sup else ("FALSIFIED" if fal else "INCONCLUSIVE")

# ------------------------------ cached unit runner ------------------------------
def run_units(unit_fn, tasks, label, flush_s=60):
    global RES
    done = set(RES["unit"]) if len(RES) else set()
    buf, t0, tl, k = [], time.time(), time.time(), 0
    for key, kw in tasks:
        if key in done: continue
        rows = unit_fn(**kw)
        for r in rows: r["unit"] = key
        buf += rows; k += 1
        if time.time() - tl > flush_s:
            RES = pd.concat([RES, pd.DataFrame(buf)], ignore_index=True); RES.to_csv(RES_PATH, index=False); buf, tl = [], time.time()
            print(f"  [{label}] {k} units done ({time.time()-t0:.0f}s)")
    if buf:
        RES = pd.concat([RES, pd.DataFrame(buf)], ignore_index=True); RES.to_csv(RES_PATH, index=False)
    print(f"  [{label}] finished: {k} new units in {time.time()-t0:.0f}s (rows cached: {len(RES)})")

def fit_seed(seed, fold, extra=0):
    return 1000 * seed + fold + 7 * extra

def seed_pivot(df, cols, val="auc"):
    d = df.copy(); d["cm"] = d[cols].astype(str).agg("_".join, axis=1)
    return d.groupby(["seed", "cm"])[val].mean().unstack("cm")
print("core library ready")

core library ready


In [5]:
# ============================== DATA ==============================
def find_mat(name):
    for root in [DATA_DIR, "/mnt/data", "/mnt/data/files", "/kaggle/input", ".", "/mnt/user-data/uploads"]:
        if root and os.path.isdir(root):
            hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
            if hits: return hits[0]
    raise FileNotFoundError(f"{name} not found. Attach it as a Kaggle Dataset or set DATA_DIR.")

def load_dataset(tag):
    path = find_mat(f"Data{tag}.mat"); M = sio.loadmat(path)
    missing = [k for k in ("Y", "SR", "SD") if k not in M]
    if missing: raise KeyError(f"{path} lacks {missing}; keys present: {[k for k in M if not k.startswith('__')]}")
    Y, SR, SD = (np.asarray(M[k], float) for k in ("Y", "SR", "SD"))
    n, m = SR.shape[0], SD.shape[0]
    if Y.shape == (m, n): Y = Y.T                                    # stored disease x drug -> drug x disease
    assert Y.shape == (n, m), f"unexpected shapes {Y.shape}, {SR.shape}, {SD.shape}"
    SR, SD = np.clip((SR + SR.T) / 2, 0, None), np.clip((SD + SD.T) / 2, 0, None)
    Y = (Y > 0).astype(float)
    D = NS(tag=tag, name=f"{tag}dataset", Y=Y, SR=SR, SD=SD, n=n, m=m)
    D.ordR, D.ordD = neighbor_order(SR), neighbor_order(SD)
    D.SgR, D.SgD = build_graph(SR, PAPER["p"]), build_graph(SD, PAPER["p"])
    LR, LD = np.diag(D.SgR.sum(1)) - D.SgR, np.diag(D.SgD.sum(1)) - D.SgD
    for nm, L in (("eigR", LR), ("eigD", LD)):
        w, V = np.linalg.eigh(L); setattr(D, nm, (np.clip(w, 0, None), V))
    D.neg_flat = np.flatnonzero(Y.ravel() == 0)
    print(f"[{D.name}] {os.path.basename(path)}: Y {Y.shape}, positives={int(Y.sum())}, density={Y.mean():.4f}")
    return D

LOADED = {t: load_dataset(t) for t in ("F", "C")}

[Fdataset] DataF.mat: Y (593, 313), positives=1933, density=0.0104
[Cdataset] DataC.mat: Y (663, 409), positives=2532, density=0.0093


## Shared configuration and diagnostic fit

The RH6–RH8 iteration caps are explicitly defined and are at least 400. The diagnostic fit records whether the update tolerance was reached before the cap. This is important because reaching the maximum iteration count is not evidence of convergence.

### Kaggle GPU execution
Enable **Settings → Accelerator → GPU (T4)** before running. Backend selection is automatic; `WNMFDDA_BACKEND=cpu` forces CPU and `WNMFDDA_BACKEND=cuda` requires CUDA. The default session budget is 10 hours (set `WNMFDDA_SESSION_BUDGET_HOURS` to change it). Each completed seed/fold unit is saved to a backend-specific CSV under `/kaggle/working`; rerun the notebook later to resume. Download the CSV from Kaggle Output before the session storage is lost. GPU updates use float32, so validate against CPU on a small smoke run before interpreting scientific results.


In [6]:
RH6_MAX_ITER = max(MAX_ITER, 400)
RH7_MAX_ITER = max(MAX_ITER, 400)
RH8_MAX_ITER = max(MAX_ITER, 400)
RH6_LAM, RH6_BETA = [0.0, 0.1, 1.0, 10.0], [0.002, 0.02, 0.2]
RH7_LAM = [0.0, 0.1, 0.3, 1.0, 3.0, 10.0]
RH7_N_PERM = 2 if PROFILE == "smoke" else 20
RH8_LAM = [0.0, 0.1, 0.3, 1.0, 3.0, 10.0]
RH_MARGIN, RH_ALPHA = 0.01, 0.05
RH_RES_PATH = RES_PATH
RH_RES = pd.read_csv(RH_RES_PATH) if os.path.exists(RH_RES_PATH) and not FORCE_RERUN else pd.DataFrame()

def _torch_nmf_fit(M, SgR, SgD, k, lam, beta, max_iter, seed, tol=1e-4):
    rng=np.random.default_rng(seed); n,m=M.shape; c=2*np.sqrt(max(float(M.mean()),1e-8)/k)
    A0=rng.random((k,n))*c+1e-6; B0=rng.random((k,m))*c+1e-6
    dev=DEVICE; dtype=torch.float32
    Mt=torch.as_tensor(np.asarray(M),device=dev,dtype=dtype); Rt=torch.as_tensor(np.asarray(SgR),device=dev,dtype=dtype); Dt=torch.as_tensor(np.asarray(SgD),device=dev,dtype=dtype)
    A=torch.as_tensor(A0,device=dev,dtype=dtype); B=torch.as_tensor(B0,device=dev,dtype=dtype)
    dR=Rt.sum(1).unsqueeze(0); dD=Dt.sum(1).unsqueeze(0); converged=False
    for it in range(1,max_iter+1):
        An=A*(B@Mt.T+lam*(A@Rt))/(beta*A+lam*A*dR+(B@B.T)@A+EPS)
        Bn=B*(An@Mt+lam*(B@Dt))/(beta*B+lam*B*dD+(An@An.T)@B+EPS)
        err=max(float(((An-A)**2).sum(0).max().item()),float(((Bn-B)**2).sum(0).max().item()))
        A,B=An,Bn
        if err<tol: converged=True; break
    if dev.type == "cuda": torch.cuda.synchronize()
    return A.detach().cpu().numpy(),B.detach().cpu().numpy(),it,converged

def _torch_nmf_fit_lap(M, LR, LD, k, lam, beta, max_iter, seed, tol=1e-4):
    rng=np.random.default_rng(seed); n,m=M.shape; c=2*np.sqrt(max(float(M.mean()),1e-8)/k)
    A0=rng.random((k,n))*c+1e-6; B0=rng.random((k,m))*c+1e-6
    dev=DEVICE; dtype=torch.float32
    Mt=torch.as_tensor(np.asarray(M),device=dev,dtype=dtype); LRt=torch.as_tensor(np.asarray(LR),device=dev,dtype=dtype); LDt=torch.as_tensor(np.asarray(LD),device=dev,dtype=dtype)
    A=torch.as_tensor(A0,device=dev,dtype=dtype); B=torch.as_tensor(B0,device=dev,dtype=dtype)
    LRp,LRn=torch.clamp(LRt,min=0),torch.clamp(-LRt,min=0); LDp,LDn=torch.clamp(LDt,min=0),torch.clamp(-LDt,min=0); converged=False
    for it in range(1,max_iter+1):
        An=A*(B@Mt.T+lam*(A@LRn))/((B@B.T)@A+beta*A+lam*(A@LRp)+EPS)
        Bn=B*(An@Mt+lam*(B@LDn))/((An@An.T)@B+beta*B+lam*(B@LDp)+EPS)
        err=max(float(((An-A)**2).sum(0).max().item()),float(((Bn-B)**2).sum(0).max().item()))
        A,B=An,Bn
        if err<tol: converged=True; break
    if dev.type == "cuda": torch.cuda.synchronize()
    return A.detach().cpu().numpy(),B.detach().cpu().numpy(),it,converged

def nmf_fit_diag(M, SgR, SgD, k, lam, beta, max_iter, seed, tol=1e-4):
    if DEVICE is not None and DEVICE.type == "cuda": return _torch_nmf_fit(M,SgR,SgD,k,lam,beta,max_iter,seed,tol)
    rng=np.random.default_rng(seed); n,m=M.shape; c=2*np.sqrt(max(M.mean(),1e-8)/k)
    A=rng.random((k,n))*c+1e-6; B=rng.random((k,m))*c+1e-6; dR,dD=SgR.sum(1)[None,:],SgD.sum(1)[None,:]; converged=False
    for it in range(1,max_iter+1):
        An=A*(B@M.T+lam*(A@SgR))/(beta*A+lam*A*dR+(B@B.T)@A+EPS)
        Bn=B*(An@M+lam*(B@SgD))/(beta*B+lam*B*dD+(An@An.T)@B+EPS)
        err=max(((An-A)**2).sum(0).max(),((Bn-B)**2).sum(0).max()); A,B=An,Bn
        if err<tol: converged=True; break
    return A,B,it,converged

def rh_run(task, unit_fn):
    global RH_RES
    if time.monotonic() >= SESSION_DEADLINE:
        print(f"{task}: session budget reached; rerun later to resume from {RH_RES_PATH}"); return
    done=set(RH_RES.loc[RH_RES.task==task,'unit'].astype(str)) if len(RH_RES) else set()
    t0=time.time(); completed=0; stopped=False
    for D in LOADED.values():
        for seed in SEEDS:
            for fold in range(N_FOLDS):
                if time.monotonic() >= SESSION_DEADLINE: stopped=True; break
                key=f"{task}_{D.tag}_s{seed}_f{fold}"
                if key in done: continue
                rows=unit_fn(D,seed,fold)
                for row in rows: row['unit']=key
                RH_RES=pd.concat([RH_RES,pd.DataFrame(rows)],ignore_index=True)
                RH_RES.to_csv(RH_RES_PATH,index=False); completed+=1
            if stopped: break
        if stopped: break
    print(task,'checkpointed units:',completed,'elapsed sec:',round(time.time()-t0,1),'total cached rows:',len(RH_RES))
    if stopped: print('Budget reached. Rerun later; completed seed/fold units will be skipped.')

def ci_seed(values): return pstats(np.asarray(values,float),alpha=RH_ALPHA)
def fit_seed_rh(seed,fold,offset=0): return 100000*seed+100*fold+offset
print('Caps:',RH6_MAX_ITER,RH7_MAX_ITER,RH8_MAX_ITER,'| profile:',PROFILE,'| backend:',BACKEND,'| cache:',RH_RES_PATH)


Caps: 400 400 400 | profile: quick | backend: torch-cuda | cache: /kaggle/working/wnmfdda_rh678_results_quick_torch_cuda.csv


---
# RH6 — Do graph regularization and factor shrinkage interact?

### E — Equations and assumptions
The objective contains a graph smoothness term scaled by `lambda` and a factor-norm penalty scaled by `beta`:

\[
\|Y'-A^TB\|_F^2+\lambda[\mathrm{Tr}(ALA^T)+\mathrm{Tr}(BL_DB^T)]+\beta(\|A\|_F^2+\|B\|_F^2).
\]

### P — Rejection hypothesis
**RH6:** the AUC effect of changing `lambda` from 0.1 to 10 is practically the same at `beta=0.002` and `beta=0.2`.

### Q — Primary contrast and rule
\[
I=[AUC(10,0.2)-AUC(0.1,0.2)]-[AUC(10,0.002)-AUC(0.1,0.002)].
\]
Average folds within each seed before calculating the 95% t interval. Margin: ±0.01 AUC. **FALSIFIED** if the interval is entirely outside the margin; **SUPPORTED for this contrast** if entirely inside; otherwise **INCONCLUSIVE**.

### V — Verification
Same outer split and test pairs for every grid cell. The interaction contrast is fixed in advance. Held-out positives are removed before reconstruction/factorization.

### L — Consequences
An interaction means regularization effects should be reported jointly at these settings; it does not by itself prove that a new architecture is needed.

In [7]:
def rh6_unit(D,seed,fold):
    sp=pair_split(D,seed,fold); rows=[]
    for lam in RH6_LAM:
        for beta in RH6_BETA:
            M=wknn_reconstruct(sp.Ytr,D)[0]
            A,B,nit,conv=nmf_fit_diag(M,D.SgR,D.SgD,PAPER['k'],lam,beta,RH6_MAX_ITER,fit_seed_rh(seed,fold,round(lam*10)+round(beta*1000)))
            met=metrics(A.T@B,sp.pos,sp.neg,True)
            rows.append(dict(task='RH6',dataset=D.tag,seed=seed,fold=fold,lam=lam,beta=beta,auc=met['auc'],aupr=met['aupr'],iterations=nit,converged=conv))
    return rows
rh_run('RH6',rh6_unit)
RH6_RESULTS=RH_RES[RH_RES.task=='RH6'].copy()
display(RH6_RESULTS.groupby(['dataset','lam','beta']).agg(auc_mean=('auc','mean'),auc_sd=('auc','std'),aupr_mean=('aupr','mean'),convergence_rate=('converged','mean'),median_iterations=('iterations','median')).reset_index())

RH6 checkpointed units: 50 elapsed sec: 138.6 total cached rows: 600


,dataset,lam,beta,auc_mean,auc_sd,aupr_mean,convergence_rate,median_iterations
0,C,0.0,0.002,0.919317,0.007661,0.223662,1.0,226.0
1,C,0.0,0.020,0.922539,0.005776,0.222933,1.0,221.0
2,C,0.0,0.200,0.932242,0.005936,0.268867,1.0,217.0
3,C,0.1,0.002,0.932527,0.005322,0.293813,1.0,211.0
4,C,0.1,0.020,0.934030,0.006508,0.299570,1.0,207.0
5,C,0.1,0.200,0.938266,0.006018,0.329677,1.0,200.0
6,C,1.0,0.002,0.942004,0.005700,0.402126,1.0,134.0
7,C,1.0,0.020,0.942444,0.005903,0.402258,1.0,125.0
8,C,1.0,0.200,0.943367,0.006206,0.408172,1.0,105.0
9,C,10.0,0.002,0.840595,0.008821,0.086101,1.0,52.0


In [8]:
rh6_out=[]
for ds in LOADED:
 d=RH6_RESULTS[RH6_RESULTS.dataset==ds]; w=d.pivot_table(index=['seed','fold'],columns=['lam','beta'],values='auc')
 need=[(10.0,.2),(.1,.2),(10.0,.002),(.1,.002)]
 if not all(k in w.columns for k in need): continue
 f=((w[(10.0,.2)]-w[(.1,.2)])-(w[(10.0,.002)]-w[(.1,.002)]))
 z=f.groupby(level='seed').mean().to_numpy(); s=ci_seed(z)
 verdict='INCONCLUSIVE'
 if np.isfinite(s['ci_lo']):
  if s['ci_lo']>RH_MARGIN or s['ci_hi'] < -RH_MARGIN: verdict='FALSIFIED'
  elif s['ci_lo']>=-RH_MARGIN and s['ci_hi']<=RH_MARGIN: verdict='SUPPORTED (tested contrast)'
 rh6_out.append(dict(hypothesis='RH6',dataset=ds,n_seeds=s['n'],mean_interaction=s['mean'],ci95_lo=s['ci_lo'],ci95_hi=s['ci_hi'],verdict=verdict))
RH6_PRIMARY=pd.DataFrame(rh6_out); display(RH6_PRIMARY)

,hypothesis,dataset,n_seeds,mean_interaction,ci95_lo,ci95_hi,verdict
0,RH6,F,5,-0.024277,-0.038559,-0.009994,INCONCLUSIVE
1,RH6,C,5,-0.013297,-0.017610,-0.008985,INCONCLUSIVE


---
# RH7 — Does the entity-aligned graph outperform node-permuted graphs?

### E — Equations and assumptions
Permuting node identities preserves the graph's weighted topology up to relabeling but breaks correspondence between graph nodes and drug/disease identities.

### P — Rejection hypothesis
**RH7:** after validation-based tuning, the true graph does not provide at least 0.01 AUC advantage over node-permuted graphs.

### Q — Primary quantity and rule
\(\Delta=AUC(\text{true})-\operatorname{mean}_b AUC(\text{permuted}_b)\). Average folds within seed. **FALSIFIED** if the 95% CI is entirely above +0.01; **SUPPORTED (advantage below margin)** if the upper bound is below +0.01; otherwise **INCONCLUSIVE**.

### V — Verification
Tune `lambda` on an inner pair-validation split drawn only from outer training data, independently for each graph condition. Use identical outer test pairs. `smoke` uses two permutations for pipeline validation; final analysis should use substantially more.

### L — Consequences
A true-graph advantage supports entity-aligned graph information. Similar performance does not prove the graph is useless; it may indicate weak graph quality or limited power.

In [9]:
def perm_graph(S,seed):
 p=np.random.default_rng(seed).permutation(S.shape[0]); return S[np.ix_(p,p)]
def rh7_score(Ytr,D,SR,SD,lam,seed):
 M=wknn_reconstruct(Ytr,D)[0]
 A,B,nit,conv=nmf_fit_diag(M,SR,SD,PAPER['k'],lam,PAPER['beta'],RH7_MAX_ITER,seed)
 return A.T@B,nit,conv
def rh7_select(Ytr,D,SR,SD,seed):
 Yin,vpos,vneg=dev_split(Ytr,np.random.default_rng(seed),'pair'); vneg=np.flatnonzero((Yin.ravel()==0)&(D.Y.ravel()==0)); best=RH7_LAM[0]; bestauc=-np.inf
 for lam in RH7_LAM:
  Sc,_,_=rh7_score(Yin,D,SR,SD,lam,seed+13); auc=metrics(Sc,vpos,vneg,False)['auc']
  if auc>bestauc: best,bestauc=lam,auc
 return best,bestauc
def rh7_unit(D,seed,fold):
 sp=pair_split(D,seed,fold); conds=[('TRUE',D.SgR,D.SgD)]
 for b in range(RH7_N_PERM):
  conds.append((f'PERM{b}',perm_graph(D.SgR,800000+seed*10000+fold*100+b),perm_graph(D.SgD,900000+seed*10000+fold*100+b)))
 rows=[]
 for j,(name,SR,SD) in enumerate(conds):
  lam,val=rh7_select(sp.Ytr,D,SR,SD,fit_seed_rh(seed,fold,200+j)); Sc,nit,conv=rh7_score(sp.Ytr,D,SR,SD,lam,fit_seed_rh(seed,fold,400+j)); met=metrics(Sc,sp.pos,sp.neg,True)
  rows.append(dict(task='RH7',dataset=D.tag,seed=seed,fold=fold,condition=name,lam=lam,val_auc=val,auc=met['auc'],aupr=met['aupr'],iterations=nit,converged=conv))
 return rows
rh_run('RH7',rh7_unit)
RH7_RESULTS=RH_RES[RH_RES.task=='RH7'].copy()
display(RH7_RESULTS.groupby(['dataset','condition']).agg(auc_mean=('auc','mean'),aupr_mean=('aupr','mean'),convergence_rate=('converged','mean'),median_iterations=('iterations','median')).reset_index())

RH7 checkpointed units: 50 elapsed sec: 1229.5 total cached rows: 1650


,dataset,condition,auc_mean,aupr_mean,convergence_rate,median_iterations
0,C,PERM0,0.924701,0.296396,1.00,187.0
1,C,PERM1,0.926287,0.307659,1.00,184.0
2,C,PERM10,0.925750,0.312225,1.00,206.0
3,C,PERM11,0.926318,0.315620,1.00,200.0
4,C,PERM12,0.925860,0.307835,1.00,199.0
5,C,PERM13,0.925149,0.304541,1.00,188.0
6,C,PERM14,0.925757,0.297725,1.00,196.0
7,C,PERM15,0.926044,0.314249,1.00,213.0
8,C,PERM16,0.926275,0.320438,1.00,196.0
9,C,PERM17,0.926687,0.309017,1.00,197.0


In [10]:
rh7_out=[]
for ds in LOADED:
 d=RH7_RESULTS[RH7_RESULTS.dataset==ds]; w=d.pivot_table(index=['seed','fold'],columns='condition',values='auc'); pc=[c for c in w.columns if str(c).startswith('PERM')]
 if 'TRUE' not in w or not pc: continue
 z=(w['TRUE']-w[pc].mean(axis=1)).groupby(level='seed').mean().to_numpy(); s=ci_seed(z)
 verdict='INCONCLUSIVE'
 if np.isfinite(s['ci_lo']):
  if s['ci_lo']>RH_MARGIN: verdict='FALSIFIED'
  elif s['ci_hi']<RH_MARGIN: verdict='SUPPORTED (advantage below margin)'
 rh7_out.append(dict(hypothesis='RH7',dataset=ds,permutations=len(pc),n_seeds=s['n'],mean_delta=s['mean'],ci95_lo=s['ci_lo'],ci95_hi=s['ci_hi'],verdict=verdict))
RH7_PRIMARY=pd.DataFrame(rh7_out); display(RH7_PRIMARY)

,hypothesis,dataset,permutations,n_seeds,mean_delta,ci95_lo,ci95_hi,verdict
0,RH7,F,20,5,0.018052,0.015819,0.020285,FALSIFIED
1,RH7,C,20,5,0.015831,0.012530,0.019133,FALSIFIED


---
# RH8 — Does normalized-Laplacian regularization change prediction?

### E — Equations and assumptions
The original implementation uses the unnormalized graph penalty. The normalized alternative uses \(L_{sym}=I-D^{-1/2}SD^{-1/2}\), with isolated nodes assigned zero penalty. This requires an objective-consistent update; substituting a normalized adjacency into the old update is not sufficient.

### P — Rejection hypothesis
**RH8:** replacing the unnormalized graph Laplacian with the normalized-Laplacian objective does not materially change held-out AUC after separate validation tuning.

### Q — Primary quantity and rule
\(\Delta_{norm}=AUC(normalized)-AUC(original)\). Average folds within seed. **FALSIFIED** if the 95% CI lies entirely outside ±0.01; **SUPPORTED (equivalent within margin)** if entirely inside; otherwise **INCONCLUSIVE**.

### V — Verification
Same outer splits and test pairs; tune lambda on inner training-side validation separately for each formulation; record convergence.

### L — Consequences
A reproducible difference makes graph normalization a meaningful modeling choice, but does not establish universal superiority.

In [11]:
def norm_laplacian(S):
 S=np.asarray(S,float); deg=S.sum(1); inv=np.zeros_like(deg); ok=deg>EPS; inv[ok]=1/np.sqrt(deg[ok])
 L=np.diag(ok.astype(float))-inv[:,None]*S*inv[None,:]; L[~ok,:]=0; L[:,~ok]=0
 return L
def nmf_fit_lap(M,LR,LD,k,lam,beta,max_iter,seed,tol=1e-4):
 # Split-gradient multiplicative updates for normalized-Laplacian objective.
 if DEVICE is not None and DEVICE.type == "cuda": return _torch_nmf_fit_lap(M,LR,LD,k,lam,beta,max_iter,seed,tol)
 rng=np.random.default_rng(seed); n,m=M.shape; c=2*np.sqrt(max(M.mean(),1e-8)/k)
 A=rng.random((k,n))*c+1e-6; B=rng.random((k,m))*c+1e-6; converged=False
 LRp,LRn=np.maximum(LR,0),np.maximum(-LR,0); LDp,LDn=np.maximum(LD,0),np.maximum(-LD,0)
 for it in range(1,max_iter+1):
  An=A*(B@M.T+lam*(A@LRn))/((B@B.T)@A+beta*A+lam*(A@LRp)+EPS)
  Bn=B*(An@M+lam*(B@LDn))/((An@An.T)@B+beta*B+lam*(B@LDp)+EPS)
  err=max(((An-A)**2).sum(0).max(),((Bn-B)**2).sum(0).max()); A,B=An,Bn
  if err<tol: converged=True; break
 return A,B,it,converged

def rh8_score(Ytr,D,lam,seed,normed):
 M=wknn_reconstruct(Ytr,D)[0]
 if normed: A,B,nit,conv=nmf_fit_lap(M,norm_laplacian(D.SgR),norm_laplacian(D.SgD),PAPER['k'],lam,PAPER['beta'],RH8_MAX_ITER,seed)
 else: A,B,nit,conv=nmf_fit_diag(M,D.SgR,D.SgD,PAPER['k'],lam,PAPER['beta'],RH8_MAX_ITER,seed)
 return A.T@B,nit,conv
def rh8_select(Ytr,D,seed,normed):
 Yin,vpos,vneg=dev_split(Ytr,np.random.default_rng(seed),'pair'); vneg=np.flatnonzero((Yin.ravel()==0)&(D.Y.ravel()==0)); best=RH8_LAM[0]; bestauc=-np.inf
 for lam in RH8_LAM:
  Sc,_,_=rh8_score(Yin,D,lam,seed+23,normed); auc=metrics(Sc,vpos,vneg,False)['auc']
  if auc>bestauc: best,bestauc=lam,auc
 return best,bestauc
def rh8_unit(D,seed,fold):
 sp=pair_split(D,seed,fold); rows=[]
 for j,normed in enumerate([False,True]):
  name='NORMALIZED' if normed else 'UNNORMALIZED'; lam,val=rh8_select(sp.Ytr,D,fit_seed_rh(seed,fold,600+j),normed)
  Sc,nit,conv=rh8_score(sp.Ytr,D,lam,fit_seed_rh(seed,fold,700+j),normed); met=metrics(Sc,sp.pos,sp.neg,True)
  rows.append(dict(task='RH8',dataset=D.tag,seed=seed,fold=fold,condition=name,lam=lam,val_auc=val,auc=met['auc'],aupr=met['aupr'],iterations=nit,converged=conv))
 return rows
rh_run('RH8',rh8_unit)
RH8_RESULTS=RH_RES[RH_RES.task=='RH8'].copy()
display(RH8_RESULTS.groupby(['dataset','condition']).agg(auc_mean=('auc','mean'),aupr_mean=('aupr','mean'),convergence_rate=('converged','mean'),median_iterations=('iterations','median')).reset_index())

RH8 checkpointed units: 50 elapsed sec: 116.4 total cached rows: 1750


,dataset,condition,auc_mean,aupr_mean,convergence_rate,median_iterations
0,C,NORMALIZED,0.944565,0.400165,1.0,96.0
1,C,UNNORMALIZED,0.942034,0.389871,1.0,118.0
2,F,NORMALIZED,0.925355,0.273206,1.0,90.0
3,F,UNNORMALIZED,0.924511,0.282772,1.0,105.0


In [12]:
rh8_out=[]
for ds in LOADED:
 d=RH8_RESULTS[RH8_RESULTS.dataset==ds]; w=d.pivot_table(index=['seed','fold'],columns='condition',values='auc')
 if not {'NORMALIZED','UNNORMALIZED'}.issubset(set(w.columns)): continue
 z=(w['NORMALIZED']-w['UNNORMALIZED']).groupby(level='seed').mean().to_numpy(); s=ci_seed(z); verdict='INCONCLUSIVE'
 if np.isfinite(s['ci_lo']):
  if s['ci_lo']>RH_MARGIN or s['ci_hi'] < -RH_MARGIN: verdict='FALSIFIED'
  elif s['ci_lo']>=-RH_MARGIN and s['ci_hi']<=RH_MARGIN: verdict='SUPPORTED (equivalent within margin)'
 rh8_out.append(dict(hypothesis='RH8',dataset=ds,n_seeds=s['n'],mean_delta=s['mean'],ci95_lo=s['ci_lo'],ci95_hi=s['ci_hi'],verdict=verdict))
RH8_PRIMARY=pd.DataFrame(rh8_out); display(RH8_PRIMARY)

,hypothesis,dataset,n_seeds,mean_delta,ci95_lo,ci95_hi,verdict
0,RH8,F,5,0.000844,-0.000900,0.002588,SUPPORTED (equivalent within margin)
1,RH8,C,5,0.002530,0.000277,0.004784,SUPPORTED (equivalent within margin)


---
# Consolidated evidence and Holm correction

The six primary comparisons are RH6/RH7/RH8 × Fdataset/Cdataset. The tables above use seed-level confidence intervals and practical-effect margins for verdicts. The following table gathers all verdicts in one place. Because equivalence tests are not identical to ordinary null-hypothesis tests, do not interpret a non-significant p-value as proof of equivalence; the CI/margin verdict is primary.

In [13]:
parts=[]
for name in ['RH6_PRIMARY','RH7_PRIMARY','RH8_PRIMARY']:
 if name in globals() and len(globals()[name]): parts.append(globals()[name].copy())
if parts:
 FINAL_PRIMARY=pd.concat(parts,ignore_index=True,sort=False)
 display(FINAL_PRIMARY)
else: print('Run RH6–RH8 cells first.')
print('Convergence audit:')
if len(RH_RES): display(RH_RES.groupby(['task','dataset']).agg(n=('auc','size'),convergence_rate=('converged','mean'),median_iterations=('iterations','median'),max_iterations=('iterations','max')).reset_index())

,hypothesis,dataset,n_seeds,mean_interaction,ci95_lo,ci95_hi,verdict,permutations,mean_delta
0,RH6,F,5,-0.024277,-0.038559,-0.009994,INCONCLUSIVE,NaN,NaN
1,RH6,C,5,-0.013297,-0.017610,-0.008985,INCONCLUSIVE,NaN,NaN
2,RH7,F,5,NaN,0.015819,0.020285,FALSIFIED,20.0,0.018052
3,RH7,C,5,NaN,0.012530,0.019133,FALSIFIED,20.0,0.015831
4,RH8,F,5,NaN,-0.000900,0.002588,SUPPORTED (equivalent within margin),NaN,0.000844
5,RH8,C,5,NaN,0.000277,0.004784,SUPPORTED (equivalent within margin),NaN,0.002530


Convergence audit:


,task,dataset,n,convergence_rate,median_iterations,max_iterations
0,RH6,C,300,1.000000,146.0,366
1,RH6,F,300,1.000000,135.5,377
2,RH7,C,525,0.998095,196.0,400
3,RH7,F,525,1.000000,177.0,366
4,RH8,C,50,1.000000,110.0,210
5,RH8,F,50,1.000000,96.5,195


## Final reporting checklist

- [ ] In Kaggle, enable **Settings → Accelerator → GPU (T4)** and verify the printed device is CUDA.
- [ ] Run smoke first and compare CPU/GPU scores and convergence; GPU uses float32 so tiny differences are expected.
- [ ] Use quick to check stability, then full only if time permits. The default session budget is 10 hours and runs resume from per-unit CSV checkpoints.
- [ ] Download the backend-specific results CSV from `/kaggle/working` after each session; working storage is not a permanent archive.
- [ ] Report profile, seed/fold counts, iteration caps, backend/device, convergence rates, AUC and AUPR.
- [ ] Keep primary contrasts and the ±0.01 AUC margin fixed before examining results.
- [ ] Increase RH7 permutations for final analysis; smoke permutations are pipeline checks only.
- [ ] State that unobserved pairs are treated as negatives.
- [ ] Do not claim a new architecture is necessary from these experiments alone; first establish a reproducible failure mode.
